# Prepare clean product-receipt history

## Goal

Convert the private sales export into one row per **customer + date + product**.

Both ordinary sales (`ПРОДАЖА`) and gifts (`ПОДАРОК`) count as products received by the customer. They remain separate in `paid_quantity` and `gift_quantity`, while `received_quantity` combines both for future recency and replenishment features.

## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
input_path = project_root / "data" / "raw" / "продажи 18.xlsx"
output_path = project_root / "data" / "interim" / "cleaned_purchases.csv"

## 2. Load, rename, and normalize columns

Customer and product codes are loaded as strings so pandas cannot interpret identifiers as numbers. Dates remain datetimes, and quantity and bonus values remain numeric.

In [ ]:
column_mapping = {
    "КлиентДляОплатыКод": "customer_id",
    "ТоварКод": "product_id",
    "Категория": "product_category",
    "БизнесЛиния": "business_line",
    "ДатаПродажи": "purchase_date",
    "Количество": "quantity",
    "Gen_ Bus_ Posting Group": "transaction_type",
    "Gen_ Prod_ Posting Group": "item_type",
    "БонусДокумента": "bonus_amount",
    "Акция": "promotion_name",
}

identifier_dtypes = {
    "КлиентДляОплатыКод": "string",
    "ТоварКод": "string",
}

df = pd.read_excel(input_path, dtype=identifier_dtypes)

missing_columns = sorted(set(column_mapping) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing expected source columns: {missing_columns}")

df = df.rename(columns=column_mapping)

text_columns = [
    "customer_id",
    "product_id",
    "product_category",
    "business_line",
    "transaction_type",
    "item_type",
    "promotion_name",
]

for column in text_columns:
    df[column] = df[column].astype("string").str.strip()

df["purchase_date"] = pd.to_datetime(df["purchase_date"], errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["bonus_amount"] = pd.to_numeric(df["bonus_amount"], errors="coerce")

print(f"Loaded {len(df):,} source rows.")
df.head(10)

## 3. Keep positive product receipts

A retained row must identify a customer, date, and product; represent an actual product (`ТОВАР`); be either a sale or gift; and have a positive quantity.

A gift is not a paid-purchase label, but it still resets the time since the customer last received the product.

In [ ]:
receipt_mask = (
    df["customer_id"].notna()
    & df["product_id"].notna()
    & df["purchase_date"].notna()
    & df["item_type"].eq("ТОВАР")
    & df["transaction_type"].isin(["ПРОДАЖА", "ПОДАРОК"])
    & df["quantity"].gt(0)
)

product_receipts = df.loc[receipt_mask].copy()

product_receipts["paid_quantity"] = product_receipts["quantity"].where(
    product_receipts["transaction_type"].eq("ПРОДАЖА"),
    0,
)
product_receipts["gift_quantity"] = product_receipts["quantity"].where(
    product_receipts["transaction_type"].eq("ПОДАРОК"),
    0,
)
product_receipts["received_quantity"] = (
    product_receipts["paid_quantity"] + product_receipts["gift_quantity"]
)

receipt_summary = (
    product_receipts
    .groupby("transaction_type", as_index=False)
    .agg(
        source_rows=("quantity", "size"),
        total_quantity=("quantity", "sum"),
    )
)
display(receipt_summary)
product_receipts.head(10)

## 4. Combine duplicate customer-date-product rows

The grouping key is `customer_id + purchase_date + product_id`. Different products on the same date remain separate. Repeated rows for the same product are combined.

Quantities are summed. `first` keeps the first non-missing category, business line, and promotion. `max` avoids multiplying a document bonus that may be repeated across several source lines.

In [ ]:
grouping_columns = ["customer_id", "purchase_date", "product_id"]

cleaned_purchases = (
    product_receipts
    .groupby(
        grouping_columns,
        as_index=False,
        sort=False,
    )
    .agg(
        paid_quantity=("paid_quantity", "sum"),
        gift_quantity=("gift_quantity", "sum"),
        received_quantity=("received_quantity", "sum"),
        product_category=("product_category", "first"),
        business_line=("business_line", "first"),
        bonus_amount=("bonus_amount", "max"),
        promotion_name=("promotion_name", "first"),
    )
    .sort_values(grouping_columns)
    .reset_index(drop=True)
)

cleaned_purchases.head(10)

## 5. Validate and save

The checks stop the notebook if duplicate keys, non-positive receipts, or inconsistent quantity totals survive. The validated table is then saved as CSV with dates written as `YYYY-MM-DD` so the result is easy to inspect in spreadsheet software.

In [ ]:
duplicate_count = int(cleaned_purchases.duplicated(grouping_columns).sum())
non_positive_receipt_count = int(cleaned_purchases["received_quantity"].le(0).sum())
quantity_mismatch_count = int(
    (
        cleaned_purchases["received_quantity"]
        - cleaned_purchases["paid_quantity"]
        - cleaned_purchases["gift_quantity"]
    ).abs().gt(1e-9).sum()
)

assert duplicate_count == 0, f"Found {duplicate_count} duplicate grouping keys."
assert non_positive_receipt_count == 0, (
    f"Found {non_positive_receipt_count} non-positive receipts."
)
assert quantity_mismatch_count == 0, (
    f"Found {quantity_mismatch_count} inconsistent quantity totals."
)

output_path.parent.mkdir(parents=True, exist_ok=True)
cleaned_purchases.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%d",
)

saved_purchases = pd.read_csv(
    output_path,
    dtype={
        "customer_id": "string",
        "product_id": "string",
        "product_category": "string",
        "business_line": "string",
        "promotion_name": "string",
    },
    parse_dates=["purchase_date"],
)
assert len(saved_purchases) == len(cleaned_purchases)
assert list(saved_purchases.columns) == list(cleaned_purchases.columns)

validation_summary = pd.Series({
    "source_rows": len(df),
    "eligible_sale_or_gift_rows": len(product_receipts),
    "cleaned_customer_date_product_rows": len(cleaned_purchases),
    "duplicate_source_rows_combined": len(product_receipts) - len(cleaned_purchases),
    "remaining_duplicate_keys": duplicate_count,
    "customers": cleaned_purchases["customer_id"].nunique(),
    "products": cleaned_purchases["product_id"].nunique(),
})

print(f"Saved validated data to {output_path}")
validation_summary

## Next step

Use `cleaned_purchases.csv` to inspect the cleaned result and build historical scoring groups and CatBoost feature rows. When loading it again, explicitly parse `purchase_date` as a date and the identifier columns as strings. For future purchase labels, use `paid_quantity > 0`; for possession, recency, and replenishment features, use `received_quantity`.